# Absolute error bound over the HEALPix longitude derivative

This notebook reproduces the Compression Lab spatial-gradient challenge for HEALPix data. Supply an original ground-truth HEALPix Zarr store and the decoded, denormalized Zarr store exported by the hyperprior inference notebook. The analysis compares their signed derivatives along the periodic longitude coordinate and reports how often the reconstruction violates the same absolute derivative-error bound.

A HEALPix array has no longitude dimension that can be rolled. The equivalent operation therefore keeps each pixel's latitude fixed and interpolates the HEALPix map at longitudes $1.25^\circ$ west and east. This is the angular displacement produced by the reference notebook's five-index offset on its $0.25^\circ$ longitude grid. Only the longitude derivative is evaluated; no latitudinal derivative or gradient magnitude is added.

To make the numerical results comparable, the notebook follows the corrected reference implementation's roll order and wrapped denominator:

$$\frac{x(\lambda+1.25^\circ)-x(\lambda-1.25^\circ)}{(2.5^\circ)\bmod 360^\circ} = \frac{x_E-x_W}{2.5^\circ}.$$

This matches the corrected challenge code. A pixel is a violation when the absolute difference between the decoded and original signed derivatives exceeds the configured bound. HEALPix interpolation and any earlier remapping can still produce small differences from a latitude-longitude calculation.

## 1. Environment and imports

Use the same `climgen` environment as the compression notebooks. This cell configures writable plotting caches and imports the numerical, HEALPix, plotting, and Zarr dependencies. No model checkpoint or GPU is required.

In [ ]:
import json
import os
import platform
import re
import warnings
from functools import lru_cache
from pathlib import Path
from urllib.parse import urlsplit

CACHE_ROOT = Path('/tmp') / f'healpix_gradient_cache_{os.getuid()}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT))

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from tqdm.auto import tqdm

display(pd.Series({
    'Python': platform.python_version(),
    'xarray': xr.__version__,
    'healpy': hp.__version__,
}).to_frame('version'))

## 2. Editable analysis configuration

Set the two Zarr locations and the absolute longitude-derivative error bound. The optional `VARIABLES` mapping follows the compression notebooks: timestep entries are decoded-store positional indices or end-exclusive ranges such as `"5-100"`; variables may be grouped under `2D` and `3D`; and `level_indices` may be one integer, a list, or `None`. Omitting timesteps uses all decoded timesteps, omitting variable groups uses all common HEALPix variables, and omitting level indices uses all levels available in the decoded store.

When the decoded export contains `source_time_index`, the selected decoded positions are matched to those exact ground-truth positions. Otherwise, time and level coordinates are matched by value. Ordering is normally discovered from metadata; leave the explicit ordering controls at `auto` unless the input lacks that metadata. `REFERENCE_LONGITUDE_INDEX_OFFSET=5` and `REFERENCE_LONGITUDE_SPACING_DEGREES=0.25` reproduce the reference dataset and should normally remain unchanged. The error-bound units are the variable's physical units per degree.

In [ ]:
GROUND_TRUTH_ZARR = os.environ.get(
    'HPX_GRADIENT_GROUND_TRUTH', '/work/bm1235/k202181/ngc4008/ngc4008_P1D_9.zarr', 
    #'HPX_GRADIENT_GROUND_TRUTH', '/p/project1/training2640/meuer1/data/nextGEMS_level9.zarr'
)
DECODED_ZARR = os.environ.get(
    'HPX_GRADIENT_DECODED', '/work/bd1560/k204233/FieldSpaceNN/notebooks/evaluations/hackathon_hyperprior/hus_only_finetune/decoded_fields.zarr'
    #'HPX_GRADIENT_DECODED', '/p/home/jusers/meuer1/jusuf/FieldSpace-compression/notebooks/evaluations/hackathon_hyperprior/hpx9_tas/decoded_fields.zarr'
)
OUTPUT_DIR = Path(os.environ.get(
    'HPX_GRADIENT_OUTPUT', './healpix_spatial_gradient_results'
))

VARIABLES = {
    'timesteps': [0],  # Delete this entry to evaluate all decoded timesteps.
    # With no groups, every common numeric HEALPix variable and all levels are used.
    # '2D': {'tas': {}},
    '2D': {'hus': {'level_indices': [-2]}},
    #'3D': {'ua': {'level_indices': [-36]}},
}

ABSOLUTE_GRADIENT_ERROR_BOUND = 1e-6
REFERENCE_LONGITUDE_INDEX_OFFSET = 5
REFERENCE_LONGITUDE_SPACING_DEGREES = 0.25
GROUND_TRUTH_ORDERING = 'auto' # 'auto', 'NESTED', or 'RING'
DECODED_ORDERING = 'auto'      # 'auto', 'NESTED', or 'RING'


## 3. Dataset-selection and alignment helpers

These helpers validate HEALPix cell counts, discover ordering metadata, flatten the optional 2-D/3-D variable groups, and align decoded time and level coordinates with the original data. The decoded export's `source_time_index` is authoritative when present. A mismatch is rejected instead of silently comparing unrelated fields.

In [ ]:
def is_remote(location):
    scheme = urlsplit(str(location)).scheme.lower()
    return bool(scheme and scheme != 'file')


def open_zarr_store(location):
    if not is_remote(location) and not Path(location).expanduser().exists():
        raise FileNotFoundError(f'Zarr store does not exist: {location}')
    return xr.open_zarr(str(location), consolidated=None)


def expand_positions(selection, size, label):
    if selection is None:
        return list(range(size))
    items = selection if isinstance(selection, (list, tuple)) else [selection]
    positions = []
    for item in items:
        if isinstance(item, (int, np.integer)) and not isinstance(item, bool):
            positions.append(int(item))
        elif isinstance(item, str) and (match := re.fullmatch(r'\s*(\d+)\s*-\s*(\d+)\s*', item)):
            start, stop = map(int, match.groups())
            if stop <= start:
                raise ValueError(f'Invalid {label} range {item!r}: stop must exceed start.')
            positions.extend(range(start, stop))
        else:
            raise ValueError(f'Invalid {label} entry {item!r}.')
    if not positions or min(positions) < 0 or max(positions) >= size:
        raise IndexError(f'{label} positions must fall in [0, {size - 1}].')
    if len(positions) != len(set(positions)):
        raise ValueError(f'{label} selection contains duplicates or overlapping ranges.')
    return positions


def normalize_level_positions(value, size, variable):
    if value is None:
        return list(range(size))
    values = [value] if isinstance(value, (int, np.integer)) else list(value)
    if not values or any(not isinstance(index, (int, np.integer)) for index in values):
        raise ValueError(f'level_indices for {variable!r} must contain integers.')
    normalized = [int(index) % size if int(index) < 0 else int(index) for index in values]
    if min(normalized) < 0 or max(normalized) >= size:
        raise IndexError(f'level_indices for {variable!r} exceed the decoded depth {size}.')
    if len(normalized) != len(set(normalized)):
        raise ValueError(f'level_indices for {variable!r} contains duplicates.')
    return normalized


def flatten_variable_configuration(configuration):
    entries = {str(key): value for key, value in configuration.items() if str(key) != 'timesteps'}
    if any(group in entries for group in ('2D', '3D')):
        unexpected = set(entries) - {'2D', '3D'}
        if unexpected:
            raise ValueError(f'Unexpected entries beside 2D/3D groups: {sorted(unexpected)}')
        flattened = {}
        for group in ('2D', '3D'):
            for variable, specification in (entries.get(group) or {}).items():
                if variable in flattened:
                    raise ValueError(f'Variable {variable!r} occurs in more than one group.')
                flattened[str(variable)] = specification or {}
        return flattened
    return {str(variable): specification or {} for variable, specification in entries.items()}


def healpix_dimension(data_array):
    candidates = []
    for dimension, size in data_array.sizes.items():
        try:
            nside = hp.npix2nside(int(size))
        except ValueError:
            continue
        if nside > 0 and nside & (nside - 1) == 0:
            priority = 0 if ('cell' in dimension.lower() or 'pixel' in dimension.lower()) else 1
            candidates.append((priority, dimension, nside))
    if not candidates:
        raise ValueError(f'{data_array.name!r} has no dimension with a valid HEALPix cell count.')
    candidates.sort()
    if len(candidates) > 1 and candidates[0][0] == candidates[1][0]:
        raise ValueError(f'Ambiguous HEALPix dimensions for {data_array.name!r}: {candidates}.')
    return candidates[0][1], candidates[0][2]


def axis_dimension(data_array, kind, excluded):
    aliases = {
        'time': {'time', 'times', 'date', 'datetime'},
        'level': {'lev', 'level', 'levels', 'plev', 'height', 'depth', 'altitude', 'model_level'},
    }[kind]
    axis = {'time': 'T', 'level': 'Z'}[kind]
    candidates = []
    for dimension in data_array.dims:
        if dimension in excluded:
            continue
        coordinate = data_array.coords.get(dimension)
        attrs = {} if coordinate is None else {str(k).lower(): str(v).lower() for k, v in coordinate.attrs.items()}
        score = 50 * (dimension.lower() in aliases) + 100 * (attrs.get('axis', '').upper() == axis)
        if kind == 'time':
            score += 100 * (attrs.get('standard_name') == 'time')
        if score:
            candidates.append((score, dimension))
    return max(candidates)[1] if candidates else None


def detect_ordering(dataset, requested, label):
    requested = str(requested).upper()
    if requested in {'NESTED', 'RING'}:
        return requested
    if requested != 'AUTO':
        raise ValueError(f'{label}_ORDERING must be auto, NESTED, or RING.')
    attribute_sets = [dataset.attrs] + [value.attrs for value in dataset.variables.values()]
    for attributes in attribute_sets:
        for key, value in attributes.items():
            key_lower, value_lower = str(key).lower(), str(value).lower()
            if 'order' in key_lower:
                if 'nest' in value_lower:
                    return 'NESTED'
                if 'ring' in value_lower:
                    return 'RING'
            if 'nest' in key_lower and isinstance(value, (bool, np.bool_)):
                return 'NESTED' if bool(value) else 'RING'
    warnings.warn(f'No HEALPix ordering metadata found for {label}; assuming NESTED.', stacklevel=2)
    return 'NESTED'


def coordinate_positions(reference, requested, label):
    positions = reference.to_index().get_indexer(pd.Index(np.asarray(requested)))
    if (positions < 0).any():
        missing = np.asarray(requested)[positions < 0]
        raise ValueError(f'{label} coordinates are absent from ground truth: {missing.tolist()}')
    return positions.tolist()

## 4. HEALPix longitude-derivative and evaluation functions

The reference analysis rolls a $0.25^\circ$ longitude grid by five indices. Here, the same $1.25^\circ$ west/east positions are sampled with bilinear HEALPix interpolation while latitude remains fixed. Ground truth and decoded maps are independently converted to RING ordering first, so differing stored orderings remain comparable. The east-minus-west numerator and positive $2.5^\circ$ wrapped denominator follow the corrected source notebook.

Missing values are rejected because interpolating across an unknown region would make the pass/fail result ambiguous. Every selected time and vertical level is evaluated independently, and only scalar metrics are retained.

In [ ]:
@lru_cache(maxsize=4)
def longitude_sample_points(nside, index_offset, grid_spacing_degrees):
    pixels = np.arange(hp.nside2npix(nside))
    theta, phi = hp.pix2ang(nside, pixels, nest=False)
    half_width_degrees = float(index_offset) * float(grid_spacing_degrees)
    if not 0 < half_width_degrees < 180:
        raise ValueError('The longitude stencil half-width must be between 0 and 180 degrees.')
    delta = np.deg2rad(half_width_degrees)
    return theta, np.mod(phi - delta, 2 * np.pi), np.mod(phi + delta, 2 * np.pi)


def differentiate_along_longitude_healpix(
    values, nside, ordering, index_offset, grid_spacing_degrees
):
    values = np.asarray(values, dtype=np.float64).squeeze()
    expected = hp.nside2npix(nside)
    if values.ndim != 1 or values.size != expected:
        raise ValueError(f'Expected one map with {expected} pixels, got {values.shape}.')
    if not np.isfinite(values).all():
        raise ValueError(f'Derivative input contains {int((~np.isfinite(values)).sum())} non-finite values.')
    ring_values = hp.reorder(values, n2r=True) if ordering == 'NESTED' else values
    theta, phi_west, phi_east = longitude_sample_points(
        int(nside), int(index_offset), float(grid_spacing_degrees)
    )
    west = hp.get_interp_val(ring_values, theta, phi_west, nest=False)
    east = hp.get_interp_val(ring_values, theta, phi_east, nest=False)

    # Match the corrected challenge: (x[i+5] - x[i-5]) divided by the
    # positive wrapped coordinate difference, here 2 * offset * spacing.
    longitude_difference = np.mod(
        2.0 * float(index_offset) * float(grid_spacing_degrees), 360.0
    )
    if longitude_difference == 0:
        raise ValueError('The wrapped reference longitude denominator is zero.')
    return (east - west) / longitude_difference


def prepare_variable_pair(ground_truth, decoded, variable, specification, timestep_selection):
    if variable not in ground_truth.data_vars or variable not in decoded.data_vars:
        raise KeyError(f'{variable!r} must be a data variable in both stores.')
    truth, reconstruction = ground_truth[variable], decoded[variable]
    truth_cell, truth_nside = healpix_dimension(truth)
    decoded_cell, decoded_nside = healpix_dimension(reconstruction)
    if truth_nside != decoded_nside:
        raise ValueError(f'{variable!r} has NSIDE {truth_nside} in ground truth and {decoded_nside} decoded.')

    decoded_time = axis_dimension(reconstruction, 'time', {decoded_cell})
    truth_time = axis_dimension(truth, 'time', {truth_cell})
    if decoded_time is None:
        if timestep_selection not in (None, [], [0]):
            raise ValueError(f'{variable!r} has no decoded time dimension.')
        decoded_time_positions = [0]
    else:
        decoded_time_positions = expand_positions(
            timestep_selection, reconstruction.sizes[decoded_time], 'timestep'
        )
        reconstruction = reconstruction.isel({decoded_time: decoded_time_positions})
        if truth_time is None:
            raise ValueError(f'{variable!r} has decoded time but no ground-truth time dimension.')
        if 'source_time_index' in decoded and decoded_time in decoded['source_time_index'].dims:
            source_positions = np.asarray(
                decoded['source_time_index'].isel({decoded_time: decoded_time_positions}).values, dtype=int
            )
            if source_positions.min() < 0 or source_positions.max() >= truth.sizes[truth_time]:
                raise IndexError('Decoded source_time_index falls outside the ground-truth store.')
            truth = truth.isel({truth_time: source_positions.tolist()})
        elif decoded_time in reconstruction.coords and truth_time in truth.coords:
            truth_positions = coordinate_positions(
                truth[truth_time], reconstruction[decoded_time].values, 'time'
            )
            truth = truth.isel({truth_time: truth_positions})
        elif truth.sizes[truth_time] == decoded.sizes[decoded_time]:
            truth = truth.isel({truth_time: decoded_time_positions})
        else:
            raise ValueError('Cannot align decoded and ground-truth time axes.')

    decoded_level = axis_dimension(reconstruction, 'level', {decoded_cell, decoded_time})
    truth_level = axis_dimension(truth, 'level', {truth_cell, truth_time})
    if decoded_level is None:
        if specification.get('level_indices') is not None:
            raise ValueError(f'{variable!r} has no decoded level dimension.')
        decoded_level_positions = [0]
        if truth_level is not None:
            raise ValueError(f'{variable!r} has a ground-truth level dimension but decoded data does not.')
    else:
        decoded_level_positions = normalize_level_positions(
            specification.get('level_indices'), reconstruction.sizes[decoded_level], variable
        )
        reconstruction = reconstruction.isel({decoded_level: decoded_level_positions})
        if truth_level is None:
            raise ValueError(f'{variable!r} has decoded levels but no ground-truth level dimension.')
        if decoded_level in reconstruction.coords and truth_level in truth.coords:
            truth_positions = coordinate_positions(
                truth[truth_level], reconstruction[decoded_level].values, 'level'
            )
            truth = truth.isel({truth_level: truth_positions})
        elif truth.sizes[truth_level] == decoded.sizes[decoded_level]:
            truth = truth.isel({truth_level: decoded_level_positions})
        else:
            raise ValueError('Cannot align decoded and ground-truth level axes.')

    allowed_truth = {truth_cell, truth_time, truth_level, None}
    allowed_decoded = {decoded_cell, decoded_time, decoded_level, None}
    if set(truth.dims) - allowed_truth or set(reconstruction.dims) - allowed_decoded:
        raise ValueError(f'{variable!r} contains unsupported dimensions after time/level selection.')
    return {
        'truth': truth, 'decoded': reconstruction, 'truth_cell': truth_cell,
        'decoded_cell': decoded_cell, 'truth_time': truth_time, 'decoded_time': decoded_time,
        'truth_level': truth_level, 'decoded_level': decoded_level,
        'time_positions': decoded_time_positions, 'level_positions': decoded_level_positions,
        'nside': truth_nside,
    }


def iter_aligned_maps(pair):
    n_times = pair['decoded'].sizes[pair['decoded_time']] if pair['decoded_time'] else 1
    n_levels = pair['decoded'].sizes[pair['decoded_level']] if pair['decoded_level'] else 1
    for time_offset in range(n_times):
        for level_offset in range(n_levels):
            truth_indexers, decoded_indexers = {}, {}
            if pair['truth_time']:
                truth_indexers[pair['truth_time']] = time_offset
                decoded_indexers[pair['decoded_time']] = time_offset
            if pair['truth_level']:
                truth_indexers[pair['truth_level']] = level_offset
                decoded_indexers[pair['decoded_level']] = level_offset
            truth_map = np.asarray(pair['truth'].isel(truth_indexers).values).squeeze()
            decoded_map = np.asarray(pair['decoded'].isel(decoded_indexers).values).squeeze()
            if truth_map.ndim != 1 or decoded_map.ndim != 1:
                raise ValueError(f'Expected one-dimensional HEALPix maps, got {truth_map.shape} and {decoded_map.shape}.')
            yield {
                'truth': truth_map, 'decoded': decoded_map,
                'timestep_position': pair['time_positions'][time_offset],
                'level_position': pair['level_positions'][level_offset],
                'time_value': (
                    pair['decoded'][pair['decoded_time']].values[time_offset]
                    if pair['decoded_time'] and pair['decoded_time'] in pair['decoded'].coords else None
                ),
                'level_value': (
                    pair['decoded'][pair['decoded_level']].values[level_offset]
                    if pair['decoded_level'] and pair['decoded_level'] in pair['decoded'].coords else None
                ),
            }


def derivative_comparison(truth_values, decoded_values, nside, truth_order, decoded_order):
    truth_derivative = differentiate_along_longitude_healpix(
        truth_values, nside, truth_order,
        REFERENCE_LONGITUDE_INDEX_OFFSET, REFERENCE_LONGITUDE_SPACING_DEGREES,
    )
    decoded_derivative = differentiate_along_longitude_healpix(
        decoded_values, nside, decoded_order,
        REFERENCE_LONGITUDE_INDEX_OFFSET, REFERENCE_LONGITUDE_SPACING_DEGREES,
    )
    signed_error = decoded_derivative - truth_derivative
    return truth_derivative, decoded_derivative, signed_error

## 5. Open and validate both stores

Only metadata is read initially. The cell finds variables present on compatible HEALPix grids in both stores, validates explicitly requested names, reports inferred ordering and resolution, and estimates how many maps the selection will evaluate. If automatic ordering falls back to `NESTED`, confirm that assumption against the data producer.

In [ ]:
if not np.isfinite(ABSOLUTE_GRADIENT_ERROR_BOUND) or ABSOLUTE_GRADIENT_ERROR_BOUND < 0:
    raise ValueError('ABSOLUTE_GRADIENT_ERROR_BOUND must be finite and non-negative.')
if not isinstance(REFERENCE_LONGITUDE_INDEX_OFFSET, (int, np.integer)) or REFERENCE_LONGITUDE_INDEX_OFFSET <= 0:
    raise ValueError('REFERENCE_LONGITUDE_INDEX_OFFSET must be a positive integer.')
if (not np.isfinite(REFERENCE_LONGITUDE_SPACING_DEGREES)
        or REFERENCE_LONGITUDE_SPACING_DEGREES <= 0):
    raise ValueError('REFERENCE_LONGITUDE_SPACING_DEGREES must be finite and positive.')

ground_truth = open_zarr_store(GROUND_TRUTH_ZARR)
decoded = open_zarr_store(DECODED_ZARR)
truth_ordering = detect_ordering(ground_truth, GROUND_TRUTH_ORDERING, 'GROUND_TRUTH')
decoded_ordering = detect_ordering(decoded, DECODED_ORDERING, 'DECODED')
variable_specs = flatten_variable_configuration(VARIABLES)

if not variable_specs:
    common = []
    for name in sorted(set(ground_truth.data_vars) & set(decoded.data_vars)):
        try:
            truth_cell, truth_nside = healpix_dimension(ground_truth[name])
            decoded_cell, decoded_nside = healpix_dimension(decoded[name])
        except ValueError:
            continue
        if truth_nside == decoded_nside and np.issubdtype(ground_truth[name].dtype, np.number):
            common.append(name)
    variable_specs = {name: {} for name in common}
if not variable_specs:
    raise ValueError('The stores have no common numeric HEALPix variables.')

geometry_rows = []
for variable in variable_specs:
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    geometry_rows.append({
        'variable': variable, 'HPX level': int(np.log2(pair['nside'])), 'NSIDE': pair['nside'],
        'selected timesteps': len(pair['time_positions']), 'selected levels': len(pair['level_positions']),
        'maps': len(pair['time_positions']) * len(pair['level_positions']),
    })
geometry = pd.DataFrame(geometry_rows).set_index('variable')
display(pd.Series({
    'Ground truth': str(GROUND_TRUTH_ZARR), 'Decoded reconstruction': str(DECODED_ZARR),
    'Ground-truth ordering': truth_ordering, 'Decoded ordering': decoded_ordering,
    'Reference longitude offset': f'±{REFERENCE_LONGITUDE_INDEX_OFFSET} × '
                                  f'{REFERENCE_LONGITUDE_SPACING_DEGREES:g}°',
    'Wrapped denominator': f'{np.mod(2 * REFERENCE_LONGITUDE_INDEX_OFFSET * REFERENCE_LONGITUDE_SPACING_DEGREES, 360):g}°',
    'Absolute error bound': f'{ABSOLUTE_GRADIENT_ERROR_BOUND:g} per degree',
    'Total maps': int(geometry['maps'].sum()),
}).to_frame('value'))
display(geometry)

## 6. Evaluate the gradient-error bound

Each selected variable/time/level map is loaded and processed independently. Just like the reference challenge, a violation is counted where the absolute difference between decoded and original signed longitude derivatives exceeds the bound. The per-map table reports the violation fraction, mean absolute error, RMSE, maximum absolute error, and derivative ranges. The summary aggregates pixel counts rather than averaging percentages, so every HEALPix cell has equal weight.

In [ ]:
metric_rows = []
for variable in tqdm(list(variable_specs), desc='Variables'):
    pair = prepare_variable_pair(
        ground_truth, decoded, variable, variable_specs[variable], VARIABLES.get('timesteps')
    )
    map_count = len(pair['time_positions']) * len(pair['level_positions'])
    for item in tqdm(iter_aligned_maps(pair), total=map_count, desc=f'{variable} maps', leave=False):
        truth_derivative, decoded_derivative, signed_error = derivative_comparison(
            item['truth'], item['decoded'], pair['nside'], truth_ordering, decoded_ordering
        )
        absolute_error = np.abs(signed_error)
        violations = absolute_error > ABSOLUTE_GRADIENT_ERROR_BOUND
        metric_rows.append({
            'variable': variable, 'timestep_position': item['timestep_position'],
            'time_value': str(item['time_value']), 'level_position': item['level_position'],
            'level_value': str(item['level_value']), 'pixel_count': signed_error.size,
            'violation_count': int(violations.sum()),
            'violation_fraction': float(violations.mean()),
            'mean_absolute_derivative_error': float(absolute_error.mean()),
            'derivative_error_rmse': float(np.sqrt(np.mean(signed_error ** 2))),
            'max_absolute_derivative_error': float(absolute_error.max()),
            'truth_derivative_min': float(truth_derivative.min()),
            'truth_derivative_max': float(truth_derivative.max()),
            'decoded_derivative_min': float(decoded_derivative.min()),
            'decoded_derivative_max': float(decoded_derivative.max()),
        })

metrics = pd.DataFrame(metric_rows)
if metrics.empty:
    raise RuntimeError('No maps were evaluated.')
total_pixels = int(metrics['pixel_count'].sum())
total_violations = int(metrics['violation_count'].sum())
summary = pd.Series({
    'Result': 'PASS' if total_violations == 0 else 'FAIL',
    'Evaluated maps': len(metrics), 'Evaluated pixels': total_pixels,
    'Violating pixels': total_violations,
    'Violation fraction': total_violations / total_pixels,
    'Worst map maximum error': metrics['max_absolute_derivative_error'].max(),
    'Absolute error bound': ABSOLUTE_GRADIENT_ERROR_BOUND,
})
display(summary.to_frame('value'))
display(metrics)

In [ ]:
gt = ground_truth.hus.isel(time=0, level_full=-2).values
dec = decoded.hus.isel(time=0, level_full=0).values

## 7. Plot one comparison

These controls affect only this visualization, so they can be changed without rerunning the complete evaluation. Positions refer to the decoded store. The first two maps share symmetric limits because the longitude derivative is signed. The third panel shows the signed decoded-minus-original derivative error with the same $\pm$ bound used by the reference challenge. All maps are plotted in RING ordering because the derivative routine normalizes both inputs to that ordering.

In [ ]:
PLOT_VARIABLE = next(iter(variable_specs))
PLOT_TIMESTEP_INDEX = 0
PLOT_LEVEL_INDEX = 0

if PLOT_VARIABLE not in variable_specs:
    raise ValueError(f'PLOT_VARIABLE must be one of {list(variable_specs)}.')
plot_specification = dict(variable_specs[PLOT_VARIABLE])
plot_cell_dimension, _ = healpix_dimension(decoded[PLOT_VARIABLE])
plot_time_dimension = axis_dimension(decoded[PLOT_VARIABLE], 'time', {plot_cell_dimension})
plot_level_dimension = axis_dimension(
    decoded[PLOT_VARIABLE], 'level', {plot_cell_dimension, plot_time_dimension}
)
if plot_level_dimension is not None:
    plot_specification['level_indices'] = [PLOT_LEVEL_INDEX]
elif PLOT_LEVEL_INDEX != 0:
    raise IndexError(f'{PLOT_VARIABLE!r} is 2-D; PLOT_LEVEL_INDEX must be 0.')
plot_pair = prepare_variable_pair(
    ground_truth, decoded, PLOT_VARIABLE, plot_specification, [PLOT_TIMESTEP_INDEX]
)
plot_item = next(iter_aligned_maps(plot_pair))
truth_derivative, decoded_derivative, signed_error = derivative_comparison(
    plot_item['truth'], plot_item['decoded'], plot_pair['nside'], truth_ordering, decoded_ordering
)
plot_violations = np.abs(signed_error) > ABSOLUTE_GRADIENT_ERROR_BOUND
combined = np.concatenate((truth_derivative, decoded_derivative))
derivative_limit = max(float(np.max(np.abs(combined))), np.finfo(float).eps)
units = decoded[PLOT_VARIABLE].attrs.get('units', '')
derivative_units = f'{units} degree⁻¹'.strip()

figure = plt.figure(figsize=(16, 4.8))
hp.mollview(
    truth_derivative, nest=False, fig=figure.number, sub=(1, 3, 1),
    title='Original longitude derivative', unit=derivative_units,
    min=-derivative_limit, max=derivative_limit, cmap='RdBu_r',
)
hp.mollview(
    decoded_derivative, nest=False, fig=figure.number, sub=(1, 3, 2),
    title=f'Decoded with {plot_violations.mean():.2%} violations', unit=derivative_units,
    min=-derivative_limit, max=derivative_limit, cmap='RdBu_r',
)
hp.mollview(
    signed_error, nest=False, fig=figure.number, sub=(1, 3, 3),
    title=f'Longitude-derivative error\nbound = ±{ABSOLUTE_GRADIENT_ERROR_BOUND:g}',
    unit=derivative_units, min=-ABSOLUTE_GRADIENT_ERROR_BOUND,
    max=ABSOLUTE_GRADIENT_ERROR_BOUND, cmap='RdBu_r',
)
figure.suptitle(
    f'{PLOT_VARIABLE} · decoded timestep {PLOT_TIMESTEP_INDEX} · level {PLOT_LEVEL_INDEX}', y=1.03
)
figure.subplots_adjust(wspace=0.08, top=0.83)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_PATH = OUTPUT_DIR / f'{PLOT_VARIABLE}_spatial_gradient.png'
figure.savefig(FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'Figure: {FIGURE_PATH.resolve()}')

## 8. Save reusable results

The final cell writes the per-map measurements and a compact JSON summary. These files contain evaluation results only; neither input store is modified. The decoded Zarr size is not treated as the compression ratio because it contains reconstructed floating-point fields rather than the entropy-coded artifacts measured by the inference notebook.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUTPUT_DIR / 'spatial_gradient_metrics.csv'
SUMMARY_PATH = OUTPUT_DIR / 'spatial_gradient_summary.json'
metrics.to_csv(METRICS_PATH, index=False)
summary_payload = {
    'ground_truth_zarr': str(GROUND_TRUTH_ZARR),
    'decoded_zarr': str(DECODED_ZARR),
    'variables': list(variable_specs),
    'ground_truth_ordering': truth_ordering, 'decoded_ordering': decoded_ordering,
    'reference_longitude_index_offset': int(REFERENCE_LONGITUDE_INDEX_OFFSET),
    'reference_longitude_spacing_degrees': float(REFERENCE_LONGITUDE_SPACING_DEGREES),
    'reference_wrapped_denominator_degrees': float(np.mod(
        2 * REFERENCE_LONGITUDE_INDEX_OFFSET * REFERENCE_LONGITUDE_SPACING_DEGREES, 360
    )),
    'absolute_gradient_error_bound': float(ABSOLUTE_GRADIENT_ERROR_BOUND),
    'result': summary['Result'], 'evaluated_maps': int(summary['Evaluated maps']),
    'evaluated_pixels': int(total_pixels), 'violating_pixels': int(total_violations),
    'violation_fraction': float(total_violations / total_pixels),
    'maximum_absolute_derivative_error': float(metrics['max_absolute_derivative_error'].max()),
}
SUMMARY_PATH.write_text(json.dumps(summary_payload, indent=2) + '\n', encoding='utf-8')
display(pd.Series({
    'Metrics CSV': str(METRICS_PATH.resolve()),
    'Summary JSON': str(SUMMARY_PATH.resolve()),
    'Comparison figure': str(FIGURE_PATH.resolve()),
}).to_frame('path'))
ground_truth.close()
decoded.close()

## Interpretation

A `PASS` means that every evaluated HEALPix cell satisfies the configured absolute bound on the signed longitude-derivative error. Because HEALPix cells have equal area, the reported violation fraction is already area weighted. The calculation matches the corrected reference code's five-cell roll on a $0.25^\circ$ grid and its positive $2.5^\circ$ denominator. Keep the offset and reference spacing fixed when comparing compression models. Do not interpret this compatibility quantity as the full physical gradient magnitude on the sphere.